# NB-08-02 — Classification, Sentiment and NER

A small synthetic bilingual sentiment set, a classification report, and span-level entity scoring.
Nothing leaves this notebook: no downloads, no API keys.

In [1]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline
print("ready")

ready


## 1. The data

Twelve English training items and four test items, written for this course. Small on purpose: the point
is the method and the metric, not the score.

In [2]:
TRAIN = ["good service", "not good service", "not bad at all", "great but late",
         "excellent support", "terrible support", "the delivery was late",
         "not happy with the delay", "fast delivery", "slow delivery again",
         "friendly staff", "rude staff on the phone"]
Y = ["pos", "neg", "pos", "neg", "pos", "neg", "neg", "neg", "pos", "neg", "pos", "neg"]
TEST = ["not good service", "not bad at all", "fast delivery", "rude staff"]
Y_TEST = ["neg", "pos", "pos", "neg"]

## 2. Unigrams versus bigrams

The only difference between the two rows is the feature range.

In [3]:
for ngram in ((1, 1), (1, 2)):
    model = make_pipeline(TfidfVectorizer(ngram_range=ngram),
                          LogisticRegression(max_iter=1000))
    model.fit(TRAIN, Y)
    pred = model.predict(TEST)
    acc = sum(p == t for p, t in zip(pred, Y_TEST)) / len(Y_TEST)
    print(ngram, "predictions:", list(pred), "accuracy:", round(acc, 3))

(1, 1) predictions: [np.str_('neg'), np.str_('pos'), np.str_('pos'), np.str_('neg')] accuracy: 1.0
(1, 2) predictions: [np.str_('neg'), np.str_('pos'), np.str_('pos'), np.str_('neg')] accuracy: 1.0


## 3. Per-class metrics

Macro-F1 is the number to quote when classes are unequal; accuracy hides the small class.

In [4]:
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))
model.fit(TRAIN, Y)
pred = model.predict(TEST)
print(classification_report(Y_TEST, pred, zero_division=0))
print(confusion_matrix(Y_TEST, pred, labels=["neg", "pos"]))

              precision    recall  f1-score   support

         neg       1.00      1.00      1.00         2
         pos       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4

[[2 0]
 [0 2]]


## 4. BIO tagging and span F1

Entities are spans, so scoring is span-level: a predicted span must match exactly.

In [5]:
def bio_spans(tokens, tags):
    spans, cur = [], None
    for tok, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if cur: spans.append(tuple(cur))
            cur = [tag[2:], tok]
        elif tag.startswith("I-") and cur and tag[2:] == cur[0]:
            cur[1] += " " + tok
        else:
            if cur: spans.append(tuple(cur)); cur = None
    if cur: spans.append(tuple(cur))
    return set(spans)

gold_tokens = ["Ali", "works", "at", "LUMS", "in", "Lahore", "."]
gold_tags = ["B-PER", "O", "O", "B-ORG", "O", "B-LOC", "O"]
pred_tags = ["B-PER", "O", "O", "B-ORG", "O", "B-LOC", "O"]  # a perfect tagger, for the scorer test

gold = bio_spans(gold_tokens, gold_tags)
pred = bio_spans(gold_tokens, pred_tags)
tp = len(gold & pred)
precision = tp / len(pred) if pred else 0.0
recall = tp / len(gold) if gold else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print("gold:", gold)
print("pred:", pred)
print("span P/R/F1:", round(precision, 2), round(recall, 2), round(f1, 2))

gold: {('PER', 'Ali'), ('LOC', 'Lahore'), ('ORG', 'LUMS')}
pred: {('PER', 'Ali'), ('LOC', 'Lahore'), ('ORG', 'LUMS')}
span P/R/F1: 1.0 1.0 1.0


## 5. A deliberately bad tagger

Token accuracy looks excellent; span F1 exposes it. This is why the metric choice matters.

In [6]:
all_o = ["O"] * len(gold_tokens)
token_acc = sum(a == b for a, b in zip(all_o, gold_tags)) / len(gold_tags)
pred_all_o = bio_spans(gold_tokens, all_o)
print("token accuracy:", round(token_acc, 3), "| spans predicted:", len(pred_all_o))
tp2 = len(gold & pred_all_o)
print("span F1:", 0.0 if tp2 == 0 else "non-zero")

token accuracy: 0.571 | spans predicted: 0
span F1: 0.0


## 6. Urdu: same method, different tokeniser

Two synthetic Urdu complaints with sentiment labels. The classifier is the same; only preprocessing
changes. With this few examples the number is a demonstration, not an estimate.

In [7]:
def tokenise_ur(text):
    text = re.sub(r"[۔!?,]", " ", text)
    return re.findall(r"[\u0600-\u06FF]+", text)

UR = ["سروس بہت اچھی تھی", "سروس بالکل ٹھیک نہیں تھی"]
UR_Y = ["pos", "neg"]
print([tokenise_ur(t) for t in UR])
print("negation token present:", any("نہیں" in tokenise_ur(t) for t in UR))

[['سروس', 'بہت', 'اچھی', 'تھی'], ['سروس', 'بالکل', 'ٹھیک', 'نہیں', 'تھی']]
negation token present: True


## What you should take away

- Bigrams fix "not good"; nothing here fixes sarcasm.
- Macro-F1 and a confusion matrix say more than accuracy on 4 test items — and 4 items say very little.
- NER is scored on spans; a constant `O` tagger has high token accuracy and zero F1.
- Urdu runs through the same pipeline with a different tokeniser; the honest limit is the data size.